![Imgur](https://i.imgur.com/acSOZRh.png)

# Laboratorio n° 1. Parte B: Entrenamiento de Redes Neuronales

**Asignatura:** Redes Neuronales Profundas 
**Bloque:** 1 — Fundamentos de Deep Learning 

---

## Introducción

En el TP1 aprendiste a operar con tensores. Ahora vas a usar esas herramientas para algo concreto: **entrenar redes neuronales**.

En este trabajo vas a:

1. Implementar el **loop de entrenamiento** de una red de clasificación de imágenes desde cero.
2. Entender qué pasa en cada paso del ciclo: forward pass, cálculo de pérdida, backward pass y actualización de pesos.
3. (Opcional) Construir un **denoising autoencoder**: una red que aprende a eliminar ruido de imágenes.

### Dataset: FashionMNIST

Vamos a trabajar con el dataset **FashionMNIST**, un conjunto estándar de 70.000 imágenes en escala de grises de 28×28 píxeles de prendas de ropa, divididas en 10 categorías:

| Etiqueta | Clase |
|:-:|---|
| 0 | T-shirt/top |
| 1 | Pantalón |
| 2 | Pulóver |
| 3 | Vestido |
| 4 | Abrigo |
| 5 | Sandalia |
| 6 | Camisa |
| 7 | Zapatilla |
| 8 | Bolso |
| 9 | Bota |

---

## Instrucciones generales

- Las celdas de **setup** ya están escritas — ejecutálas sin modificar.
- Completá el código en las celdas marcadas con `# Tu código aquí`.
- Respondé las preguntas de análisis en las celdas de texto marcadas con .

---
## Ejercicios


In [ ]:
# Imports
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils import data
import matplotlib.pyplot as plt

print(f"PyTorch {torch.__version__}")

### Ejercicio 1 — Implementar `load_data_fashion_mnist()`

Completá la función que descarga el dataset FashionMNIST y crea los DataLoaders.

**Pistas:**
- Usá `transforms.ToTensor()` para la transformación.
- Usá `torchvision.datasets.FashionMNIST()` con `download=True`.
- Creá los DataLoaders con `data.DataLoader()`, con `shuffle=True` para entrenamiento.


In [ ]:
def load_data_fashion_mnist(batch_size):
    """
    Descarga y prepara el dataset FashionMNIST.

    Parámetros:
    batch_size (int): cantidad de imágenes por lote

    Retorna:
    train_iter: DataLoader de entrenamiento (60.000 imágenes)
    test_iter: DataLoader de prueba (10.000 imágenes)
    """
    # Tu código aquí


### Ejercicio 2 — Implementar `accuracy()`

Completá la función que cuenta cuántas predicciones fueron correctas.

**Pistas:**
- Si `y_hat` tiene más de una columna, usá `argmax` para obtener la clase predicha.
- Compará las predicciones con las etiquetas y sumá los aciertos.


In [ ]:
def accuracy(y_hat, y):
    """
    Cuenta las predicciones correctas.

    Parámetros:
    y_hat: tensor de predicciones — si tiene más de una columna,
           se toma la clase con mayor puntuación (argmax).
    y: tensor de etiquetas reales (enteros)

    Retorna:
    float: cantidad de predicciones correctas
    """
    # Tu código aquí


### Ejercicio 3 — Implementar `train()`

Completá el ciclo de entrenamiento que ejecuta múltiples épocas.

**Pistas:**
- En cada época, llamá a `train_epoch()` y `test_accuracy()` (que vas a implementar más adelante).
- Acumulá las métricas en una lista de tuplas `(epoch, loss, train_acc, test_acc)`.


In [ ]:
def train(net, train_iter, test_iter, loss, num_epochs, updater):
    """
    Entrena la red durante num_epochs épocas completas.

    Parámetros:
    net: el modelo a entrenar
    train_iter: DataLoader de entrenamiento
    test_iter: DataLoader de prueba
    loss: función de pérdida
    num_epochs: cantidad de épocas de entrenamiento
    updater: optimizador ya configurado

    Retorna:
    metrics: lista de tuplas (epoch, loss, train_acc, test_acc)
    """
    # Tu código aquí


### Ejercicio 4 — Definir el modelo, la pérdida y el optimizador

Completá el código para crear:
1. Una red `nn.Sequential` con `nn.Flatten()` y una capa `nn.Linear(784, 10)`.
2. La función de pérdida `nn.CrossEntropyLoss(reduction='none')`.
3. El optimizador `torch.optim.SGD` con `lr=0.1`.


In [ ]:
# Modelo base: red de una capa (regresión softmax)
# Este modelo es Sequential con:
# 1. Flatten: convierte las imágenes de (28×28) en vectores de 784 elementos
# 2. Linear(784, 10): una capa densa que produce 10 puntuaciones (una por clase)


net = #completar

# Inicializamos los pesos con valores pequeños de distribución normal



# Función de pérdida CrossEntropyLoss y optimizador SGD con lr=0.1
# La función de pérdida (CrossEntropyLoss) aplica Softmax internamente.

loss = # completar
trainer = # completar

print(net)

---
## El ciclo de entrenamiento

Todo el entrenamiento de una red neuronal en PyTorch sigue siempre el mismo patrón de cuatro pasos, que se repite para cada lote de datos:

```
1. FORWARD PASS: pasar las entradas por el modelo → obtener predicciones
2. LOSS: comparar predicciones con etiquetas reales → calcular el error
3. BACKWARD PASS: propagar el error hacia atrás → calcular gradientes
4. UPDATE: usar los gradientes para ajustar los pesos
```

Este ciclo se repite para todos los lotes del dataset. Recorrer *todo* el dataset una vez se llama **época**.


### Ejercicio 5 — Implementar `train_epoch()`

**Objetivo:** Implementar el entrenamiento de una época completa.

Tenés que completar la función `train_epoch()`, que recibe el modelo, los datos de entrenamiento, la función de pérdida y el optimizador, y debe:

1. Iterar sobre todos los lotes del DataLoader (`train_iter`). Cada iteración devuelve un lote `(X, y)` donde:
- `X`: tensor de imágenes del lote — forma `(batch_size, 1, 28, 28)`
- `y`: tensor de etiquetas del lote — forma `(batch_size,)`
2. Para cada lote, seguir los **4 pasos del ciclo de entrenamiento**:
- **Forward:** `y_hat = net(X)` → predicciones del modelo
- **Loss:** `l = loss(y_hat, y)` → calcular la pérdida
- **Paso previo al backward:** limpiar los gradientes del paso anterior con `updater.zero_grad()`
- **Backward:** `l.mean().backward()` → calcular los gradientes
- **Update:** `updater.step()` → actualizar los pesos
3. Acumular la pérdida total y los aciertos para calcular las métricas de la época.
4. Retornar la pérdida promedio y el accuracy de entrenamiento.

> **Pistas:**
> - Llamá a `updater.zero_grad()` **antes** de hacer `.backward()` para borrar los gradientes del lote anterior. Si no lo hacés, los gradientes se acumulan y el entrenamiento no funciona.
> - Una vez terminada la época, el accuracy de entrenamiento es: `total_aciertos / total_ejemplos`
> - La pérdida promedio es: `perdida_total / total_ejemplos`
> - Para calcular los aciertos usá la función `accuracy()` que ya está definida.

In [ ]:
def train_epoch(net, train_iter, loss, updater):
    """
    Entrena la red durante una sola época.

    Parámetros:
    net: el modelo a entrenar
    train_iter: DataLoader de entrenamiento
    loss: función de pérdida
    updater: optimizador ya configurado

    Retorna:
    L: pérdida promedio de la época
    Acc: accuracy de entrenamiento de la época
    """
    net.train() # indica al modelo que estamos en modo entrenamiento


    perdida_total = 0.0 # suma de todas las pérdidas del lote
    aciertos_total = 0.0 # suma de predicciones correctas
    n_total = 0 # total de ejemplos procesados
    # ── BUCLE DE ENTRENAMIENTO ────────────────────────────────────────────────
    for X, y in train_iter:
    # Tu código aquí

    # ── Paso 1: FORWARD PASS ─────────────────────────────────────────────

    # ── Paso 2: CALCULAR LA PÉRDIDA ──────────────────────────────────────

    # ── Paso 3: LIMPIAR GRADIENTES ───────────────────────────────────────

    # ── Paso 4: BACKWARD PASS ────────────────────────────────────────────
    # .mean() convierte el vector de pérdidas en un escalar.


    # ── Paso 5: ACTUALIZAR PARÁMETROS ────────────────────────────────────

    # ── Paso 6: ACUMULAR METRICAS ────────────────────────────────────────

    # ── Paso 7: RETORNAR LAS MÉTRICAS ────────────────────────────────────



**Pregunta de análisis:**

¿Por qué es necesario llamar a `updater.zero_grad()` antes de cada `backward()`? ¿Qué pasaría si no lo hiciéramos?

*(Escribí tu respuesta acá)*

### Ejercicio 6 — Implementar `test_accuracy()`

**Objetivo:** Implementar la evaluación del modelo sobre los datos de prueba.

La función `test_accuracy()` mide qué tan bien generaliza el modelo a datos que **no vio durante el entrenamiento**. Para hacer esto:

1. Iterar sobre todos los lotes del DataLoader de prueba (`test_iter`).
2. Para cada lote, obtener las predicciones del modelo.
3. Acumular los aciertos y el total de ejemplos.
4. Retornar el accuracy total.

> **Pistas:**
> - Durante la evaluación, no necesitamos calcular gradientes. Usá el bloque `with torch.no_grad():` para desactivarlos.
> - Antes del loop, llamá a `net.eval()` para poner el modelo en modo evaluación. Esto afecta capas como Dropout (que en evaluación no apaga neuronas). En esta red no hay Dropout, pero es buena práctica hacerlo siempre.

In [ ]:
def test_accuracy(net, test_iter):
    """
    Evalúa el accuracy del modelo sobre los datos de prueba.
    
    Parámetros:
    net: el modelo a evaluar
    test_iter: DataLoader de prueba
    
    Retorna:
    TestAcc: accuracy de prueba (entre 0 y 1)
    """
    # Tu código aquí


**Pregunta de análisis:**

¿Por qué desactivamos el cálculo de gradientes durante la evaluación con `torch.no_grad()`? ¿Qué recursos ahorra?

*(Escribí tu respuesta acá)*

### Ejercicio 7 — Entrenar y visualizar

**Objetivo:** Usar las funciones implementadas para entrenar efectivamente la red y analizar las curvas de entrenamiento.

**Pasos:**
1. Cargar el dataset con `load_data_fashion_mnist(batch_size=256)`.
2. Llamar a `train()` para entrenar el modelo durante **10 épocas**.
3. Graficar las curvas usando el código provisto más abajo.

> **Cómo usar `train()`:**
> La función `train()` ya la implementaste en el Ejercicio 3. Recibe:
> - `net`: el modelo (ya definido en el Ejercicio 4)
> - `train_iter`, `test_iter`: los dos DataLoaders
> - `loss`: la función de pérdida (ya definida en el Ejercicio 4)
> - `num_epochs`: cantidad de épocas
> - `updater`: el optimizador (ya definido en el Ejercicio 4)
>
> Devuelve una lista de tuplas `(epoch, loss, train_acc, test_acc)` que podés guardar en una variable.

In [ ]:
# Tu código aquí: cargar datos y entrenar


In [ ]:
# Código de graficación — ejecutar sin modificar
# Esta función espera recibir la lista 'metrics' que devuelve train().

def plot_metrics(metrics, titulo="Entrenamiento"):
    """
    Grafica las curvas de entrenamiento.
    
    Uso:
    metrics = train(net, train_iter, test_iter, loss, 10, trainer)
    plot_metrics(metrics)
    
    Parámetros:
    metrics: lista de tuplas (epoch, loss, train_acc, test_acc)
    tal como la devuelve la función train()
    """
    epochs = [m[0] for m in metrics]
    losses = [m[1] for m in metrics]
    train_a = [m[2] for m in metrics]
    test_a = [m[3] for m in metrics]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(epochs, losses, 'b-o', label='Pérdida')
    ax1.set_title('Pérdida de entrenamiento')
    ax1.set_xlabel('Época')
    ax1.set_ylabel('Pérdida')
    ax1.grid(True)
    ax1.legend()
    
    ax2.plot(epochs, train_a, 'g-o', label='Train accuracy')
    ax2.plot(epochs, test_a, 'r-o', label='Test accuracy')
    ax2.set_title('Accuracy')
    ax2.set_xlabel('Época')
    ax2.set_ylabel('Accuracy')
    ax2.set_ylim([0, 1])
    ax2.grid(True)
    ax2.legend()
    
    plt.suptitle(titulo)
    plt.tight_layout()
    plt.show()
    
    # Llamar a plot_metrics con tus métricas:
    # plot_metrics(metrics) ← reemplazá 'metrics' por el nombre de tu variable

**Pregunta de análisis:**

Observando las curvas:
1. ¿El modelo converge? ¿Cómo se ve eso en el gráfico de pérdida?
2. ¿Hay diferencia entre el train accuracy y el test accuracy? ¿Qué indicaría si esa diferencia fuera muy grande?

*(Escribí tu respuesta acá)*

---
## Fin del Laboratorio 1b

Completaste la primera parte del laboratorio de entrenamiento. En el Laboratorio 1c vas a construir un denoising autoencoder.